## NBA 3.1: copy a finished run from the volume into Unity Catalog tables (step 4, readiness plan section 10)

Copies the 26 run tables of `COPY_RUN` (default `full_2026_11_13`, the run of record) into `kerendia_nba_3_1`, with the names `00_cohort_config` uses in `STORAGE_MODE = "uc"`:

| Layer | Volume folder | Tables |
| --- | --- | --- |
| `pull` | `dev_pipeline/query_output/<run>/` | 4 (`rx_fills_tagged` is a scratch pull, not copied) |
| `pj` | `dev_pipeline/notebook_output/<run>/patient_journey/` | 11 |
| `hcp` | `dev_pipeline/notebook_output/<run>/hcp_claims/` | 11 (`_stage/` = temporary tables, not copied) |

* Copy = `DEEP CLONE` of the Delta folder (same files, same partitions); nothing is recomputed. The volume folders are not changed.
* Check = per table: same columns and types, same partition columns, same row count, same order-independent hash of all values.
* Then the run is added to `run_registry` and made the run of record (views `hcp_*`).
* Rerun after a stop: `SKIP_EXISTING = True` does not copy tables again that already exist.

In [0]:
%run ./00_cohort_config

In [0]:
import os
COPY_RUN = "full_2026_11_13"                          # run of record (job 1010893614518587, run 338478597334222)
COPY_RUN_NOTE = "Copied from the volume by 00_copy_run_to_uc (job 1010893614518587, run 338478597334222)"
SKIP_EXISTING = True

# This notebook works with the UC names and metadata of STORAGE_MODE = "uc" for that run; the sources are read by path.
STORAGE_MODE, RUN_NAME, VALIDATION_MODE = "uc", COPY_RUN, False

def _datasets(root, depth=0):
    """Delta folders under root (one level per subfolder), without _stage/ (temporary tables)."""
    out = []
    for e in sorted(os.listdir(root)):
        p = root + e
        if not os.path.isdir(p) or e.startswith((".", "_")):
            continue
        if os.path.isdir(p + "/_delta_log"):
            out.append(p)
        elif depth < 2:
            out += _datasets(p + "/", depth + 1)
    return out

_src = [p for r in (NBA31_QUERY_OUTPUT + COPY_RUN + "/", NBA31_RUN_OUTPUT + COPY_RUN + "/") for p in _datasets(r)]
_src = [p for p in _src if os.path.basename(p) not in _TEMP_QUERY_TABLES]   # rx_fills_tagged: scratch pull
COPY = pd.DataFrame({"SOURCE": _src, "TABLE": [path_to_table(p + "/") for p in _src]})
COPY["NAME"] = COPY.TABLE.map(_bare_table_name)
COPY["LAYER"] = COPY.NAME.str[len(COPY_RUN) + 1:].str.split("_").str[0]
assert not any(_is_path(t) for t in COPY.TABLE), "Some folders have no table name"
assert COPY.TABLE.is_unique, "Two folders map to the same table"
assert len(COPY) == 26 and COPY.LAYER.value_counts().to_dict() == {"pj": 11, "hcp": 11, "pull": 4}, COPY.LAYER.value_counts().to_dict()
display(COPY[["LAYER", "NAME", "SOURCE"]])
print(f"{len(COPY)} tables to copy for {COPY_RUN}")

In [0]:
# DEEP CLONE copies the files of the current Delta version (same data, same partitions). Comment and tags: the defaults of
# write_delta (run, layer, notebook, old path) + dataset and the source run.
import time
for r in COPY.itertuples():
    if SKIP_EXISTING and table_exists(r.TABLE):
        print(f"skip (exists) {r.NAME}")
        continue
    _t0 = time.time()
    spark.sql(f"CREATE OR REPLACE TABLE {r.TABLE} DEEP CLONE delta.`{r.SOURCE}`")
    _c, _tags = _default_metadata(r.SOURCE + "/", r.TABLE)
    set_uc_table_metadata(r.TABLE, comment=f"{_c}. {COPY_RUN_NOTE}",
                          tags={**_tags, "dataset": r.NAME[len(COPY_RUN) + 1 + len(r.LAYER) + 1:], "source": "volume copy"})
    print(f"copied {r.NAME} in {time.time() - _t0:,.0f} s")

In [0]:
# Independent of the copy: each folder and table is read again. Same columns + types, same partition columns, same row
# count, and the same sum of a hash of every row (all columns; order-independent, counts duplicates).
def _fingerprint(df):
    _h = F.xxhash64(*[F.col(f"`{c}`") for c in df.columns]).cast("decimal(38,0)")
    _r = df.agg(F.count(F.lit(1)).alias("N"), F.sum(_h).alias("H")).first()
    return _r["N"], _r["H"]

checks = []
for r in COPY.itertuples():
    _fd, _td = spark.read.format("delta").load(r.SOURCE), spark.table(r.TABLE)
    _types = [(f.name, f.dataType.simpleString()) for f in _fd.schema] == [(f.name, f.dataType.simpleString()) for f in _td.schema]
    _pf = spark.sql(f"DESCRIBE DETAIL delta.`{r.SOURCE}`").first()["partitionColumns"]
    _pt = spark.sql(f"DESCRIBE DETAIL {r.TABLE}").first()["partitionColumns"]
    (_nf, _hf), (_nt, _ht) = _fingerprint(_fd), _fingerprint(_td)
    checks.append({"TABLE": r.NAME, "COLUMNS": len(_td.columns), "ROWS_FOLDER": _nf, "ROWS_TABLE": _nt,
                   "TYPES_EQUAL": _types, "PARTITIONS": ",".join(_pt), "PARTITIONS_EQUAL": list(_pf) == list(_pt),
                   "HASH_EQUAL": _hf == _ht})
    print(f"checked {r.NAME}: {_nt:,} rows")
checks = pd.DataFrame(checks)
checks["OK"] = checks.TYPES_EQUAL & checks.PARTITIONS_EQUAL & checks.HASH_EQUAL & (checks.ROWS_FOLDER == checks.ROWS_TABLE)
display(checks)
assert checks.OK.all(), f"Copy check failed: {checks[~checks.OK].TABLE.tolist()}"
print(f"All {len(checks)} tables equal their folders")

In [0]:
# Only after the check above passed: add the run to run_registry and point the views hcp_* at its HCP tables.
assert "checks" in globals() and len(checks) == len(COPY) and checks.OK.all(), "Run the check cell first (all OK)"
register_run("SUCCESS", note=COPY_RUN_NOTE)
set_run_of_record(COPY_RUN)
display(spark.table(RUN_REGISTRY_TABLE))
_views = [r.tableName for r in spark.sql(f"SHOW VIEWS IN `{UC_CATALOG}`.`{UC_SCHEMA}`").collect()]
print(f"Views: {sorted(_views)}")